# Airbnb Calendar Data Cleaning with Pandas
This notebook profiles, cleans, validates, and analyzes Airbnb calendar data. Upload `calendar.csv` when prompted if it is not already in the working directory.

In [ ]:
from pathlib import Path
import zipfile
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

DATA_DIR = Path('data')
OUTPUT_DIR = Path('output')
DATA_DIR.mkdir(exist_ok=True)
OUTPUT_DIR.mkdir(exist_ok=True)
print('Working directory:', Path.cwd())

## 1. Locate or upload the CSV
In Colab, upload the CSV or ZIP from your computer. Locally, place the file in `data/calendar.csv`.

In [ ]:
csv_path = DATA_DIR / 'calendar.csv'
if not csv_path.exists():
    try:
        from google.colab import files
        uploaded = files.upload()
        for name in uploaded:
            p = Path(name)
            if p.suffix.lower() == '.zip':
                with zipfile.ZipFile(p) as z:
                    candidates = [n for n in z.namelist() if n.lower().endswith('.csv')]
                    if not candidates:
                        raise FileNotFoundError('The ZIP contains no CSV file.')
                    chosen = next((n for n in candidates if 'calendar' in Path(n).name.lower()), candidates[0])
                    csv_path.write_bytes(z.read(chosen))
                    break
            elif p.suffix.lower() == '.csv':
                csv_path.write_bytes(p.read_bytes())
                break
    except ImportError:
        pass
if not csv_path.exists():
    raise FileNotFoundError('Place calendar.csv in data/ or upload it in Colab.')
print('Using:', csv_path)

## 2. Load and inspect the data

In [ ]:
df = pd.read_csv(csv_path)
print('Shape:', df.shape)
display(df.head())
display(df.info())
display(df.describe(include='all').T)
display(df.isna().sum().rename('missing_count').to_frame())

## 3. Standardize columns and remove exact duplicates

In [ ]:
df.columns = (df.columns.astype(str).str.strip().str.lower()
             .str.replace(r'[^a-z0-9]+', '_', regex=True).str.strip('_'))
before = len(df)
df = df.drop_duplicates().copy()
print('Exact duplicate rows removed:', before - len(df))
print('Columns:', df.columns.tolist())
required = {'listing_id', 'date', 'available', 'price'}
missing_columns = required - set(df.columns)
if missing_columns:
    raise ValueError(f'Missing expected columns: {missing_columns}')

## 4. Convert data types

In [ ]:
df['date'] = pd.to_datetime(df['date'], errors='coerce')
availability_map = {'t': True, 'f': False, 'true': True, 'false': False}
df['available'] = df['available'].astype('string').str.strip().str.lower().map(availability_map).astype('boolean')
price_text = df['price'].astype('string').str.strip()
price_text = price_text.str.replace(r'[$,£€]', '', regex=True).str.replace(r'\\s+', '', regex=True)
df['price'] = pd.to_numeric(price_text.replace({'': pd.NA, 'nan': pd.NA, 'None': pd.NA}), errors='coerce')
display(df.dtypes.to_frame('dtype'))
display(df.isna().sum().rename('missing_count').to_frame())

## 5. Validate rows and save rows for review
Missing prices on unavailable dates may be expected. They are reported, not automatically filled.

In [ ]:
review_mask = df['listing_id'].isna() | df['date'].isna() | df['available'].isna() | (df['price'].lt(0).fillna(False))
review_rows = df.loc[review_mask].copy()
print('Rows flagged for review:', len(review_rows))
print('Missing prices when unavailable:', int((df['available'].eq(False) & df['price'].isna()).sum()))
review_rows.to_csv(OUTPUT_DIR / 'calendar_rows_for_review.csv', index=False)

## 6. Add calendar features and summarize availability

In [ ]:
df['year'] = df['date'].dt.year
df['month'] = df['date'].dt.month
df['month_name'] = df['date'].dt.month_name()
df['day_of_week'] = df['date'].dt.day_name()
df['is_weekend'] = df['date'].dt.dayofweek >= 5
monthly = (df.dropna(subset=['date']).groupby(['year','month','month_name'], as_index=False)
           .agg(total_rows=('listing_id','size'), available_rows=('available','sum'),
                mean_price=('price','mean'), median_price=('price','median')))
monthly['availability_rate_pct'] = 100 * monthly['available_rows'] / monthly['total_rows']
display(monthly.head(12))
monthly.to_csv(OUTPUT_DIR / 'monthly_summary.csv', index=False)

## 7. Explore prices and flag possible outliers
The IQR rule flags candidates for review; it does not prove a value is wrong.

In [ ]:
valid_prices = df.loc[df['price'].notna() & df['price'].ge(0), 'price']
if not valid_prices.empty:
    q1, q3 = valid_prices.quantile([0.25, 0.75])
    iqr = q3 - q1
    lower, upper = q1 - 1.5 * iqr, q3 + 1.5 * iqr
    outliers = df.loc[df['price'].lt(lower) | df['price'].gt(upper)].copy()
    print(f'IQR bounds: {lower:.2f} to {upper:.2f}; candidates: {len(outliers)}')
    outliers.to_csv(OUTPUT_DIR / 'potential_price_outliers.csv', index=False)
    display(outliers.head())
    df['price'].dropna().plot(kind='hist', bins=50, title='Distribution of calendar prices')
    plt.xlabel('Price')
    plt.show()
else:
    outliers = pd.DataFrame()
    print('No valid numeric prices to analyze.')

## 8. Export cleaned data and data-quality summary

In [ ]:
quality = pd.DataFrame({
    'metric': ['rows_after_deduplication','columns','unique_listings','invalid_dates','missing_availability','missing_prices','negative_prices','rows_flagged_for_review'],
    'value': [len(df), len(df.columns), df['listing_id'].nunique(dropna=True), int(df['date'].isna().sum()), int(df['available'].isna().sum()), int(df['price'].isna().sum()), int(df['price'].lt(0).fillna(False).sum()), len(review_rows)]
})
quality.to_csv(OUTPUT_DIR / 'data_quality_summary.csv', index=False)
df.to_csv(OUTPUT_DIR / 'cleaned_calendar.csv', index=False)
display(quality)
print('Output files:')
for p in sorted(OUTPUT_DIR.glob('*.csv')):
    print('-', p, f'({p.stat().st_size:,} bytes)')

## Conclusion
Review the quality summary and flagged records before downstream use. In particular, missing prices on unavailable dates can be expected, and price outliers should not be deleted without domain context.